<h2>Download Initial Dependencies</h2>

In [1]:
import numpy as np
import pandas as pd
import json
import math
from tqdm import tqdm
from qiskit.quantum_info import SparsePauliOp, Operator, Pauli, Clifford
import qiskit.qasm2 as q2
from qiskit import QuantumCircuit

from heisenbergUtilities import *

In [2]:
from qiskit_aer import AerSimulator
simulator = AerSimulator()

print(simulator.available_devices())

('CPU', 'GPU')


Generate the linked list of all operations that take place in the inverse circuit

In [2]:
class LinkedListNode:
    def __init__(self, name="Initial", depth=0, count=0,  next=None, prev=None):
        self.value = name
        self.depth = depth
        self.next = next
        self.prev = prev
        self.count = count

class LinkedList:
    def __init__(self):
        self.head = None

    def append(self, name, depth):
        new_node = LinkedListNode(name, depth)
        if not self.head:
            self.head = new_node
            return
        last_node = self.head
        while last_node.next:
            last_node = last_node.next
        last_node.next = new_node
        new_node.prev = last_node

    def mark(self, depth):
        current_node = self.head
        while current_node:
            if current_node.depth == depth:
                current_node.count += 1
                return
            current_node = current_node.next

    def reset(self):
        current_node = self.head
        while current_node:
            current_node.count = 0
            current_node = current_node.next

<h2>Download MQT Benchmark circuits.</h2>

We'll start with DJ, RealAmpRandom, and GHZ since SB-QOPS caught those mutants 100% of the time for 5 qubit circuits



<h2>Generate mutants.</h2>

We'll start with mutants that add an unnecessary gate, then add mutants that replace a certain gate later

<h2> Define our CUT </h2>

This is where we will use SB-QOPS on a provided circuit and its mutants

For a single mutant, it took about 2 minutes to generate a 20 test suite (10 fail, 10 pass) on an RTX 4090 SUPER. 

There are 113 mutants for the DJ algorithm in use: 226 minutes or 3 3/4 hours

There are 89 mutants for the GHZ algorithm in use: 178 minutes or about 3 hours

There are 125 mutants for the RealAmpRandom algorithm in use: 250 minutes or just over 4 hours

BUT this cell should only need to be run once per algorithm with mutants under test. After it saves to the csv file at the end, this cell can be commented out as to not accidentally run it again. 

In the future if we want to test this methodology on higher qubit circuits or other algorithms, we'll likely either want to reduce the number of tests in the suite or the number of mutants we're analyzing

In [ ]:
ORIGINAL_QASM = "dj_5_qubits.qasm"
ORIGINAL_QASM_FOLDER = "SBFL_circuits/MQTBench/"
QASM_MUTANT_FOLDER = "SBFL_circuits/QMutBenchMutants/Mutants_dj_5_qubits/"
QUBITS = 5
Runs = 10
algo = "1+1"

correct_circuit = load_program(ORIGINAL_QASM, ORIGINAL_QASM_FOLDER).copy().inverse()
correct_list = LinkedList()
depth = 1

#Append each gate to the linked list
for instruction in correct_circuit.data:
    correct_list.append(instruction.name + " " + str(depth), depth)
    depth += 1


u 3


In [ ]:
import QOPS as qops
from QOPS_test import get_compact_program_specification_Z, get_mutants
from pathlib import Path

ga_result = pd.DataFrame(columns=['Program',"path_to_mutant",'catch_avg','avg_fitness','failing_testcases', 'passing_testcases'])
program_history = {}


# program_list = [x for x in os.listdir('benchmarkFilteration/benchmark2/') if x.split(".")[0].split("_")[-1] == QUBITS]
# for program_name in tqdm(program_list):

#program_specification = The compact program specification. SB-QOPS needs this to generate failing test cases
program_specification = get_compact_program_specification_Z(correct_circuit, shots=20000)

#mutants = A python list of qiskit QuantumCircuit variables with a mutation in them
mutants = []
mutants_names = []
root = Path(QASM_MUTANT_FOLDER)
for qasm_mutant in root.rglob("*.qasm"):
    mutants.append(load_program(qasm_mutant.name, QASM_MUTANT_FOLDER))
    mutants_names.append(qasm_mutant.name)

for mutant_index,mutant in enumerate(mutants):
    tester = qops.Circuit_Tester(CUT=mutant)
    tester.set_applicable_families_Z(program_specification)
    mutants_per_run = []
    fitness_per_run = []
    failing_testcases_per_run = []
    history_per_run = []

    print("NOW RUNNING TO FIND FAILING TESTS")
    #For a predefined number of attempts, try to find a set of failing test cases (output is above predefined tolerance)
    for runs in range(Runs):
        killed = 0
        pauli = {}
        fitness = 0
        for i in range(len(tester.applicable_families)):
            best_function,testcase, history = tester.run_mealoneplusone(i, 80)
            if best_function >0.1:
                killed = 1
                pauli = testcase
                fitness = best_function
                break
        mutants_per_run.append(killed)
        failing_testcases_per_run.append(pauli)
        fitness_per_run.append(fitness)
        history_per_run.append(history)

    avg_score = np.mean(mutants_per_run)
    avg_fitness = np.mean(fitness_per_run)

    #Here is our new, modified algorithm from the SB-QOPS method that attempts to find passing test cases as well. We'll need them for SBFL
    passing_testcases_per_run = []

    print("NOW RUNNING TO FIND PASSING TESTS")
    for runs in range(Runs):
        pauli = {}
        fitness = 0
        for i in range(len(tester.applicable_families)):
            best_function, testcase, history = tester.run_reverse_mealoneplusone(i,80)
            if best_function < 0.1:
                pauli = testcase
                break
        passing_testcases_per_run.append(pauli)

    #Replace static program file with "program_name" when ready to test fully
    """
    ga_result: A pandas dataframe with the following columns
        Program: Name of the mutant quantum circuit file
        Path_To_Mutant: Path to the mutant file
        Catch_Avg: The average number of times the mutant was identified by SB-QOPS
        Avg_Fitness: The average fitness of the search algorithm used. Higher usually indicates higher Catch_Avg
        Failing_Testcases: A list of dicts indicating the set of Pauli strings and the weights that should be applied to generate a failing test case
        Passing_Testcases: A list of dicts indicating the set of Pauli strings and the weights that should be applied to generate a passing test case
    """
    ga_result = pd.concat([pd.DataFrame([[mutants_names[mutant_index],QASM_MUTANT_FOLDER,avg_score,avg_fitness,json.dumps(failing_testcases_per_run), json.dumps(passing_testcases_per_run)]],columns=ga_result.columns),ga_result],ignore_index=True)
    ga_result.to_csv(f'dj_ga_result', index=False)

2026/06/29 11:20:21 AM, INFO, mealpy.evolutionary_based.ES.OriginalES: OriginalES(epoch=80, pop_size=5, lamda=0.75)
2026/06/29 11:20:22 AM, INFO, mealpy.evolutionary_based.ES.OriginalES: >>>Problem: P, Epoch: 1, Current best: 3.5785350405719645, Global best: 3.5785350405719645, Runtime: 0.05405 seconds


NOW RUNNING TO FIND FAILING TESTS


2026/06/29 11:20:22 AM, INFO, mealpy.evolutionary_based.ES.OriginalES: >>>Problem: P, Epoch: 2, Current best: 3.5785350405719645, Global best: 3.5785350405719645, Runtime: 0.06000 seconds
2026/06/29 11:20:22 AM, INFO, mealpy.evolutionary_based.ES.OriginalES: >>>Problem: P, Epoch: 3, Current best: 3.5785350405719645, Global best: 3.5785350405719645, Runtime: 0.06273 seconds
2026/06/29 11:20:22 AM, INFO, mealpy.evolutionary_based.ES.OriginalES: >>>Problem: P, Epoch: 4, Current best: 3.5785350405719645, Global best: 3.5785350405719645, Runtime: 0.05990 seconds
2026/06/29 11:20:22 AM, INFO, mealpy.evolutionary_based.ES.OriginalES: >>>Problem: P, Epoch: 5, Current best: 5.080534639822231, Global best: 5.080534639822231, Runtime: 0.15681 seconds
2026/06/29 11:20:22 AM, INFO, mealpy.evolutionary_based.ES.OriginalES: >>>Problem: P, Epoch: 6, Current best: 5.080534639822231, Global best: 5.080534639822231, Runtime: 0.07859 seconds
2026/06/29 11:20:22 AM, INFO, mealpy.evolutionary_based.ES.Origi

NOW RUNNING TO FIND PASSING TESTS


2026/06/29 11:21:11 AM, INFO, mealpy.evolutionary_based.ES.OriginalES: >>>Problem: P, Epoch: 2, Current best: 0.13179277245860904, Global best: 0.13179277245860904, Runtime: 0.05710 seconds
2026/06/29 11:21:11 AM, INFO, mealpy.evolutionary_based.ES.OriginalES: >>>Problem: P, Epoch: 3, Current best: 0.13179277245860904, Global best: 0.13179277245860904, Runtime: 0.06321 seconds
2026/06/29 11:21:11 AM, INFO, mealpy.evolutionary_based.ES.OriginalES: >>>Problem: P, Epoch: 4, Current best: 0.13179277245860904, Global best: 0.13179277245860904, Runtime: 0.06036 seconds
2026/06/29 11:21:11 AM, INFO, mealpy.evolutionary_based.ES.OriginalES: >>>Problem: P, Epoch: 5, Current best: 0.13179277245860904, Global best: 0.13179277245860904, Runtime: 0.05685 seconds
2026/06/29 11:21:11 AM, INFO, mealpy.evolutionary_based.ES.OriginalES: >>>Problem: P, Epoch: 6, Current best: 0.13179277245860904, Global best: 0.13179277245860904, Runtime: 0.05573 seconds
2026/06/29 11:21:11 AM, INFO, mealpy.evolutionary_

...

NOW RUNNING TO FIND PASSING TESTS


2026/06/29 11:26:11 AM, INFO, mealpy.evolutionary_based.ES.OriginalES: >>>Problem: P, Epoch: 2, Current best: 0.15961453726624963, Global best: 0.15961453726624963, Runtime: 0.05986 seconds
2026/06/29 11:26:11 AM, INFO, mealpy.evolutionary_based.ES.OriginalES: >>>Problem: P, Epoch: 3, Current best: 0.15961453726624963, Global best: 0.15961453726624963, Runtime: 0.06345 seconds
2026/06/29 11:26:11 AM, INFO, mealpy.evolutionary_based.ES.OriginalES: >>>Problem: P, Epoch: 4, Current best: 0.010044718488804838, Global best: 0.010044718488804838, Runtime: 0.05618 seconds
2026/06/29 11:26:11 AM, INFO, mealpy.evolutionary_based.ES.OriginalES: >>>Problem: P, Epoch: 5, Current best: 0.010044718488804838, Global best: 0.010044718488804838, Runtime: 0.05600 seconds
2026/06/29 11:26:11 AM, INFO, mealpy.evolutionary_based.ES.OriginalES: >>>Problem: P, Epoch: 6, Current best: 0.010044718488804838, Global best: 0.010044718488804838, Runtime: 0.05958 seconds
2026/06/29 11:26:11 AM, INFO, mealpy.evoluti

NOW RUNNING TO FIND FAILING TESTS


2026/06/29 11:26:59 AM, INFO, mealpy.evolutionary_based.ES.OriginalES: >>>Problem: P, Epoch: 3, Current best: 3.3162389582967515, Global best: 3.3162389582967515, Runtime: 0.05491 seconds
2026/06/29 11:26:59 AM, INFO, mealpy.evolutionary_based.ES.OriginalES: >>>Problem: P, Epoch: 4, Current best: 3.8574160278637866, Global best: 3.8574160278637866, Runtime: 0.05406 seconds
2026/06/29 11:26:59 AM, INFO, mealpy.evolutionary_based.ES.OriginalES: >>>Problem: P, Epoch: 5, Current best: 3.8574160278637866, Global best: 3.8574160278637866, Runtime: 0.05275 seconds
2026/06/29 11:27:00 AM, INFO, mealpy.evolutionary_based.ES.OriginalES: >>>Problem: P, Epoch: 6, Current best: 3.8574160278637866, Global best: 3.8574160278637866, Runtime: 0.05478 seconds
2026/06/29 11:27:00 AM, INFO, mealpy.evolutionary_based.ES.OriginalES: >>>Problem: P, Epoch: 7, Current best: 3.8574160278637866, Global best: 3.8574160278637866, Runtime: 0.05507 seconds
2026/06/29 11:27:00 AM, INFO, mealpy.evolutionary_based.ES.O

KeyboardInterrupt: 

ga_result is a slightly altered pandas Dataframe with similar structure found in SB-QOPS's results folder. The only differences are changing the column "Program" to be the name of the mutant file instead of the original quantum circuit and changing "Mutant" to be the path to the mutant file instead of being an arbitrary index value

Now we want to run SBFL using Heisenberg evolution trees on each circuit placed in ga_result

Evolution analysis tends to take about 5 minutes for 10 failing tests, so about 10 minutes for 20 total tests

So about 1130 minutes for DJ, or 18.8 hours to evolve and analyze all 113 DJ mutants

About 890 minutes for GHZ, or 14.8 hours to evolve and analyze all 89 GHZ mutants

About 1250 minutes for RealAmpRandom, or 20.8 hours to evolve and analyze all 125 RealAmpRandom mutants

This is something to note in the final paper. Regardless of accuracy this methodology takes a long time to run. If results are promising, then we'll want to look into the tradeoffs between EXAM and hyperparameters to speed this up. Primarily the atol, max_terms, and search_step parameters in the evolve_pauli_exact method in HeisenbergUtilities.py


<h2>Heisenberg Evolutions</h2>

In [ ]:
from heisenbergTree import *

sbfl_result = pd.DataFrame(columns=['Program','path_to_mutant','exam_score'])

#For each mutant of a circuit, run the SBFL implementation
for mutant, path in zip(ga_result.loc[:,"Program"], ga_result.loc[:,"path_to_mutant"]):
    print("Now evolving the following mutant: ", mutant)

    #Extract the raw test cases found for each mutant
    desired_failing_testcases = ga_result.loc[(ga_result["Program"] == mutant), ["failing_testcases"]]
    desired_passing_testcases = ga_result.loc[(ga_result["Program"] == mutant), ["passing_testcases"]]
    tested_mutant += 1
    raw_failing_testcases = desired_failing_testcases["testcases"].values[0].split("}")
    raw_passing_testcases = desired_passing_testcases["testcases"].values[0].split("}")

    #Remove empty tests
    raw_failing_testcases = remove_null_tests(raw_failing_testcases)
    raw_passing_testcases = remove_null_tests(raw_passing_testcases)

    circuit_inverse = load_program(mutant,path).copy().inverse()  # Invert to track backward evolution of the operator

    #Create the linked list of operations in the inverse circuit
    operation_list = LinkedList()
    depth = circuit_inverse.size()

    #Append each gate to the linked list
    operation_list.append("Initial", depth)
    for instruction in circuit_inverse.data:
        depth += 1
        operation_list.append(instruction.name + " " + str((circuit_inverse.size()-depth)), circuit_inverse.size()-depth)

    #Create list of nodes in linked list. Somewhere down the road remove the linked list implementation. It's unnecessary and giving me a headache
    node_list = []
    node = operation_list.head
    while node:
        node_list.append(node)
        node = node.next

    #Perform Pauli propagation (Heisenberg evolution) for all test cases. Returns a dataframe with all counts for all operations
    global_frame = heisenberg_evolve(circuit_inverse, operation_list, raw_failing_testcases, pass_fail = "fail")
    global_frame = heisenberg_evolve(circuit_inverse, operation_list, raw_passing_testcases, pass_fail = "pass")

    global_frame = global_frame.drop(["Initial"],axis=1)

    tarantula_scores = tarantula(global_frame)
    ochiai_scores = ochiai(global_frame)

    print("Tarantula Scores: ", tarantula_scores)
    print("Ochiai Scores: ", ochiai_scores)
    
    # Here is where analysis of the methodology begins. 
    # We first extract the position of the erroneous gate by comparing it to the original MQT gate
    # NOTE: This method is intended for single-added gates for now. Other types of mutants will require other methods later
    error_gate_location = find_erroneous_gate(operation_list, correct_list)
    
    gates_observed = 0
    exam_score = 0
    for col_name, col_date in ochiai_scores.items():
        gates_observed += 1

        #Extract depth from column name
        gate_depth = int(col_name.split()[1])

        if gate_depth == error_gate_location:
            exam_score = gates_observed/circuit_inverse.size()
            break

    sbfl_result = pd.concat([pd.DataFrame([[mutants_names[mutant_index],QASM_MUTANT_FOLDER,exam_score]],columns=sbfl_result.columns),sbfl_result],ignore_index=True)
    sbfl_result.to_csv(f'dj_sbfl_result', index=False)
    




    
    

<h2>Optimization Test and One-Shot Example</h2>

Using the pauli_prop package with marginally small atol and including max_terms/search_step yielded the same results in Ochiai as the old implementation, but ran twice as fast

In [9]:
from pauli_prop import propagate_through_operator

def evolve_pauli_exact(pauli_label, gate):
    """Return exact Pauli expansion after conjugation"""
    evolution = propagate_through_operator(pauli_label, gate, atol=1e-4, frame='h', max_terms = 80, search_step=3)

    return {
        p.to_label(): c for p, c in zip(evolution.paulis, evolution.coeffs)
    }

def load_program(name,path):
    qc = QuantumCircuit.from_qasm_file("{}/{}".format(path,name))
    qc.remove_final_measurements()
    if len(qc.clbits) > 0:
        for i in range(len(qc.clbits)):
            qc.measure(i, i)
    else:
        qc.measure_all()
    qc.remove_final_measurements()
    return qc.copy()

circuit_under_test = load_program("qnn_indep_qiskit_5.qasm", "benchmarkFilteration/benchmark2/")
circuit_inverse = circuit_under_test.copy().inverse()
output_from_csv = pd.read_csv("results/1+1_result5.csv")
desired_testcases = output_from_csv.loc[(output_from_csv["Program"] == "qnn_indep_qiskit_5.qasm") & (output_from_csv["mutant"] == 2), ["testcases"]]
raw_testcases = desired_testcases["testcases"].values[0].split("}")

for raw_idx, raw in enumerate(raw_testcases):
    if raw == ']':
        print("Caught one!", raw)
        raw_testcases.pop(raw_idx)

first = True

operation_list = LinkedList()
depth = 0
operation_list.append("Initial", depth)
for instruction in circuit_inverse.data:
    depth += 1
    operation_list.append(instruction.name + " " + str((circuit_inverse.size()-depth)), depth)


node_list = []
node = operation_list.head
while node:
    node_list.append(node)
    node = node.next

global_circuit_frame = pd.DataFrame({}, columns = [node.value for node in node_list] + ["pass/fail"], dtype=np.float64)

#For every provided test case, clean the string, convert it into a dictionary, and perform Pauli propagation
for test in tqdm(raw_testcases):

    #Cleaning up the test case string from the csv file to convert it into a dictionary
    test = test.replace("{", "")
    test = test.replace("]", "")
    test = test.replace("[", "")
    if not first:
        test = test[2:]
    test = "{" +test + "}"
    test = json.loads(test)

    #We want to keep track of the number of strings are in a specific test case and weight the results so that SBFL isn't biased by tests with more strings in them
    num_strings = 0
    
    #For each string specified in the test case, perform Pauli propagation through the circuit and store the results
    for pauli_string, pauli_coeff in zip(test.keys(), test.values()):

        #print("The coefficient related to string ", pauli_string, " is ", pauli_coeff)
        num_strings += 1

        # Define the initial Pauli operator to track
        # Qiskit layout is right-to-left: 'IX' means X acts on Qubit 0, I acts on Qubit 1
        initial_pauli = SparsePauliOp(pauli_string)  
        prev_pauli = initial_pauli
        depth = 1
        num_qubits = circuit_inverse.num_qubits

        transition_graph = {}  # step -> list of (from, to, coeff)

        #Define the current layer of the evolution tree, starting with the initial Pauli having a probability of 100% to appear
        current_layer = {
            initial_pauli.paulis[0].to_label(): 1.0  # Start with the initial Pauli having probability 1
        }

        #For each gate in the quantum circuit, propagate backwards. The circuit is already inversed 
        for step_idx, (instruction, qargs, cargs) in enumerate(circuit_inverse.data, start=1):

            #Define operation being performed
            gate_type = instruction.name 
            gate_op = Operator(instruction)

            #Create a do-nothing operator, then add the desired gate to the do-nothing to only act on the desired qubits
            q_indices = [qarg._index for qarg in qargs]
            full_gate_op = Operator.from_label("I" * num_qubits).compose(gate_op, q_indices)

            new_layer = {}
            edges = []

            #For each Pauli in the current layer of the evolution tree, apply the desired operator to the desired qubits
            for pauli_in, prob_in in current_layer.items():

                evolved_dict = evolve_pauli_exact(SparsePauliOp(pauli_in), SparsePauliOp.from_operator(full_gate_op))

                #For each resulting Pauli string, calculate the probability of the inverse circuit reaching this state
                #This will be used later to weight the counts for SBFL
                for pauli_out, coeff_out in evolved_dict.items():

                    total_prob = prob_in * ((coeff_out.real)**2)

                    # accumulate node weights
                    new_layer[pauli_out] = (new_layer.get(pauli_out, 0) + total_prob)

                    # store exact edge
                    edges.append({
                        "from": pauli_in,
                        "to": pauli_out,
                        "weight": coeff_out,
                        "probability": total_prob
                    })

            #Simplify numerical noise, we don't care about states that have such a low chance of occuring that it's negligible
            current_layer = {
                pauli: prob for pauli, prob in new_layer.items() if abs(prob) > 1e-4
            }

            transition_graph[step_idx] = {
                "gate_type": gate_type,
                "edges": edges
            }

        #Start at first gate in the inverse circuit
        checked_gate = operation_list.head.next
        idx = 1

        #For each gate in the circuit
        while checked_gate:

            #For each transition through that gate
            for edge in transition_graph[idx]["edges"]:
                
                #If the previous Pauli string evolved into a new Pauli string, 
                #add the probability of that new string occuring to that gate's count
                if edge["from"] != edge["to"]:
                    checked_gate.count += edge["probability"]
            
            #We weight the count by the coefficient related to the Pauli string that SB-QOPS provides
            checked_gate.count *= abs(pauli_coeff)
            idx += 1
            checked_gate = checked_gate.next

    first = False
    node = operation_list.head
    gate_list = []
    while node:
        gate_list.append(node.count/num_strings)
        node = node.next
    gate_list.append("fail")
    global_circuit_frame = pd.concat([global_circuit_frame, pd.DataFrame([gate_list], columns=global_circuit_frame.columns)], ignore_index=True)
    operation_list.reset()

global_circuit_frame
    


Caught one! ]


  0%|          | 0/10 [00:00<?, ?it/s]C:\Users\Noste\AppData\Local\Temp\ipykernel_11928\2176267172.py:87: DeprecationWarning: Treating CircuitInstruction as an iterable is deprecated legacy behavior since Qiskit 1.2, and will be removed in Qiskit 3.0. Instead, use the `operation`, `qubits` and `clbits` named attributes.
  for step_idx, (instruction, qargs, cargs) in enumerate(circuit_inverse.data, start=1):
  0%|          | 0/10 [00:05<?, ?it/s]


KeyboardInterrupt: 

In [11]:
global_circuit_frame = global_circuit_frame.drop(["Initial"],axis=1)
global_circuit_frame

,ry 83,ry 82,ry 81,ry 80,ry 79,cx 78,cx 77,cx 76,cx 75,ry 74,...,cx 8,p 7,cx 6,u2 5,cx 4,p 3,cx 2,u2 1,u2 0,pass/fail
0,0.052186,0.004368,0.010047,0.000275,0.007467,0.148672,0.084994,0.089513,0.037638,0.000983,...,0.023479,0.000824,0.024280,0.021784,0.023606,0.000418,0.024388,0.022891,0.023314,fail
1,0.001850,0.000171,0.000986,0.000006,0.000858,0.003900,0.005093,0.003410,0.002230,0.000060,...,0.001259,0.000037,0.001302,0.001244,0.001259,0.000024,0.001307,0.001261,0.001278,fail
2,0.013023,0.000028,0.006187,0.000052,0.001940,0.029497,0.040375,0.004933,0.008813,0.000196,...,0.010266,0.000265,0.010673,0.010148,0.010409,0.000189,0.010865,0.010399,0.010345,fail
3,0.039363,0.001439,0.013459,0.000159,0.002609,0.084688,0.096941,0.043673,0.042283,0.001185,...,0.019586,0.000663,0.020124,0.019051,0.018694,0.000545,0.019514,0.016803,0.018376,fail
4,0.050620,0.004756,0.012186,0.000237,0.004852,0.130455,0.091587,0.090261,0.036256,0.001028,...,0.016363,0.000534,0.016676,0.016043,0.015055,0.000369,0.015592,0.014523,0.014753,fail
5,0.091008,0.005786,0.013180,0.000157,0.014767,0.094150,0.075803,0.082961,0.055952,0.001483,...,0.039612,0.001001,0.041096,0.037897,0.041190,0.000727,0.042922,0.040216,0.040462,fail
6,0.068710,0.010837,0.016515,0.000231,0.015427,0.132371,0.103442,0.170591,0.068410,0.001790,...,0.037618,0.000854,0.039061,0.035696,0.038984,0.000611,0.040669,0.037950,0.040069,fail
7,0.026206,0.001407,0.006043,0.000091,0.001882,0.049482,0.034463,0.026059,0.016621,0.000489,...,0.004802,0.000162,0.004871,0.004359,0.004566,0.000116,0.004685,0.004419,0.004315,fail
8,0.009940,0.000028,0.000056,0.000039,0.000645,0.020964,0.001094,0.001289,0.003879,0.000126,...,0.002090,0.000090,0.002145,0.001763,0.002036,0.000030,0.002100,0.001970,0.002030,fail
9,0.008869,0.002125,0.003771,0.000080,0.001048,0.042550,0.026471,0.041078,0.009059,0.000262,...,0.004593,0.000145,0.004720,0.004486,0.004468,0.000117,0.004646,0.004344,0.004354,fail


<h2>Tarantula Algorithm</h2>

NOTE: This algorithm will not run correctly until SB-QOPS is configured to return a selection of passing tests on top of the failing tests it already provides

In [12]:
num_fail_tests = len(global_circuit_frame[global_circuit_frame["pass/fail"] == "fail"])
num_pass_tests = len(global_circuit_frame[global_circuit_frame["pass/fail"] == "pass"])
fail_counts = global_circuit_frame[global_circuit_frame["pass/fail"] == "fail"].agg(["sum"]).drop(["pass/fail"], axis=1)
pass_counts = global_circuit_frame[global_circuit_frame["pass/fail"] == "pass"].agg(["sum"]).drop(["pass/fail"], axis=1)

tarantula_scores = (fail_counts/num_fail_tests)/((fail_counts/num_fail_tests)+(pass_counts/num_pass_tests))
tarantula_scores

,ry 83,ry 82,ry 81,ry 80,ry 79,cx 78,cx 77,cx 76,cx 75,ry 74,...,cx 9,cx 8,p 7,cx 6,u2 5,cx 4,p 3,cx 2,u2 1,u2 0
sum,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


<h2>Ochiai Algorithm</h2>

NOTE: This algorithms performance is also degraded until SB-QOPS also returns a selection of passing tests

In [15]:
pass_counts = global_circuit_frame[global_circuit_frame["pass/fail"] == "pass"].agg(["sum"]).drop(["pass/fail"], axis=1)
fail_counts = global_circuit_frame[global_circuit_frame["pass/fail"] == "fail"].agg(["sum"]).drop(["pass/fail"], axis=1)
num_fail_tests = len(global_circuit_frame[global_circuit_frame["pass/fail"] == "fail"])

ochiai_scores = fail_counts/np.sqrt(num_fail_tests*(fail_counts+pass_counts))
ochiai_scores = ochiai_scores[ochiai_scores.iloc[0].sort_values(ascending=False).index]
ochiai_scores


,cx 78,cx 77,cx 76,cx 70,cx 72,cx 58,u2 57,cx 60,cx 66,ry 83,...,p 39,p 45,p 7,p 17,p 32,p 42,p 10,p 3,p 20,ry 80
sum,0.271427,0.236699,0.235323,0.20484,0.198422,0.195701,0.192855,0.19095,0.190249,0.190203,...,0.022054,0.021427,0.021388,0.020708,0.020645,0.019876,0.018517,0.017737,0.016471,0.011518
